# PythonParser lexer — interactive playground

Try out the incremental Python lexer + the `PythonParser` syntactic-validity potential.

**Kernel:** select the `latent` conda env (it has `genlm-control`, `genlm-grammar`, `lark`, `interegular`).

The lexer maps a character stream to Python *terminal classes* (`NAME`, `DEF`, `LPAR`, `_NEWLINE`, `_INDENT`, ...). The potential then asks a terminal-level grammar whether that stream is a valid Python prefix / complete program.

In [ ]:
from genlm.control.potential.built_in.python_parser import (
    PythonParser, _PythonLexer, _build_grammar,
)
import numpy as np

larkstuff, cfg = _build_grammar()
lexer = _PythonLexer(larkstuff)
# Byte vocabulary so we can feed arbitrary text (one token per byte).
parser = PythonParser([bytes([i]) for i in range(256)])

def ctx(s):
    return [bytes([b]) for b in s.encode('utf-8')]

print('grammar terminals:', len(cfg.V), '| rules:', len(list(cfg)))

## 1. Lex a string into terminal classes

`lexer.lex(text, eof)` returns:
- `ok` — `False` on a hard lexical/indentation error,
- `terminals` — committed terminal-class names,
- `open_classes` — what the trailing in-progress token could still become (only when `eof=False`).

Use `eof=False` to lex a *prefix* (trailing token left open) and `eof=True` to lex a *complete* program (trailing token force-committed, closing `_NEWLINE`/`_DEDENT`s added).

In [ ]:
def show_lex(text, eof=False):
    res = lexer.lex(text, eof=eof)
    print(f'text  : {text!r}')
    print(f'eof   : {eof}')
    print(f'ok    : {res.ok}')
    print(f'terms : {res.terminals}')
    print(f'open  : {sorted(res.open_classes)}')

show_lex('def g(df):\n    return df.iloc[0]\n', eof=True)

### Maximal munch, keywords vs identifiers, indentation
Edit these and re-run to probe the lexer's behaviour.

In [ ]:
show_lex('a == b')          # '==' is one token (__ANON_19), not two '='
print()
show_lex('def define = 1')   # 'def' -> DEF keyword; 'define' -> NAME
print()
show_lex('x = [\n  1,\n  2,\n]\n', eof=True)  # newlines inside [] are joined

In [ ]:
# An open trailing token: 'retur' could still become RETURN or an identifier.
show_lex('def g():\n    retur')

## 1b. Maximal munch & incomplete tokens (incl. strings)

**Maximal munch:** at each position the lexer advances *every* terminal recogniser in lockstep and commits the **longest** prefix that some recogniser accepts. So `==` is one token, not two `=`; `define` is one `NAME`, not the keyword `def` + `ine`.

**Incomplete trailing token (the key case for decoding, e.g. an unclosed string):** when the input ends mid-token (`eof=False`), the lexer does *not* commit it. Instead it returns the set of **terminal classes that partial character sequence could still become** — every recogniser still alive (could reach an accept with more input) plus any already accepting. That set is `open_classes`. The potential then accepts the prefix iff at least one of those classes is a legal next terminal for the grammar.

In [ ]:
# Incomplete string: the closing quote hasn't arrived yet.
show_lex("x = 'abc")        # open_classes -> {'STRING'}: still inside a string body
print()
show_lex('y = """unclosed')  # open_classes -> {'LONG_STRING'}
print()
# Numbers and operators are ambiguous mid-token too:
show_lex('x = 3')            # {'DEC_NUMBER','FLOAT_NUMBER','IMAG_NUMBER'}: 3, 3.5, 3j ...
print()
show_lex('if x =')           # {'EQUAL','__ANON_19', ...}: '=' could still become '=='

In [ ]:
# Maximal munch in action: longest match wins, ties resolved by keyword priority.
show_lex('a == b', eof=True)     # one '==' token (__ANON_19)
print()
show_lex('a = = b', eof=True)    # two separate EQUAL tokens
print()
show_lex('returns = 1', eof=True)  # 'returns' is NAME, not RETURN + 's'

## 2. Score with the potential (top-level `await`)
`prefix` = can this extend to valid Python?  `complete` = is this a complete valid program?

In [ ]:
text = 'def g(df):\n    return df.iloc[0]'
print('prefix  finite:', np.isfinite(await parser.prefix(ctx(text))))
print('complete finite:', np.isfinite(await parser.complete(ctx(text))))

In [ ]:
# Full inspector: lex + scores + the grammar's allowed-next terminal set.
async def inspect(text, eof=False):
    res = lexer.lex(text, eof=eof)
    print(f'text          : {text!r}')
    print(f'terminals     : {res.terminals}')
    print(f'open_classes  : {sorted(res.open_classes)}')
    print(f'prefix finite : {np.isfinite(await parser.prefix(ctx(text)))}')
    print(f'complete finite: {np.isfinite(await parser.complete(ctx(text)))}')
    if res.ok and res.terminals:
        logw = await parser._parser.logw_next(res.terminals)
        allowed = [t for t in parser._parser.vocab_eos if np.isfinite(logw[t])]
        print(f'allowed next  : {allowed}')

await inspect('def g(df):')

## 3. Per-character incremental prefix validity
Walk the string one character at a time; mark where the prefix stops being valid Python (this is exactly the pruning signal the twist gives SMC).

In [ ]:
async def walk(text):
    for k in range(1, len(text) + 1):
        ok = np.isfinite(await parser.prefix(ctx(text[:k])))
        mark = ' ' if ok else 'X'
        print(f'{mark} {text[:k]!r}')

await walk('x = = 1')   # the second '=' makes it doomed

## 4. Your turn
Paste any snippet (e.g. a DS-1000 answer) and inspect it.

In [ ]:
await inspect('result = df.groupby(\'a\').size()', eof=True)

## 5. Cross-check vs CPython `tokenize`, and the memoization speedup

`compare(src)` lexes a *complete* program both ways so you can eyeball that our maximal-munch + indentation matches CPython's own tokenizer (the reference we test against). Our lexer reports Lark terminal *classes* (`DEF`, `NAME`, `LPAR`, `_INDENT`, ...); `tokenize` lumps keywords under `NAME` and operators under `OP`, but the structure lines up.

In [ ]:
import io, tokenize

def compare(src):
    res = lexer.lex(src, eof=True)
    print('our terminals:')
    print('  ', res.terminals)
    print('\nCPython tokenize:')
    keep = {tokenize.NAME, tokenize.NUMBER, tokenize.STRING, tokenize.OP,
            tokenize.INDENT, tokenize.DEDENT, tokenize.NEWLINE}
    for tok in tokenize.generate_tokens(io.StringIO(src).readline):
        if tok.type in keep:
            print(f'   {tokenize.tok_name[tok.type]:8} {tok.string!r}')

compare('def g(df):\n    return df.iloc[0]\n')

In [ ]:
# Memoization speedup: incremental decoding (cache kept) vs re-lexing each step.
import time
prog = 'def g(df, List):\n    return df.iloc[List]\n\nresult = g(df.copy(), List)'

parser.clear_cache()
t0 = time.perf_counter()
for k in range(1, len(prog) + 1):
    await parser.prefix(ctx(prog[:k]))
memo = time.perf_counter() - t0

t0 = time.perf_counter()
for k in range(1, len(prog) + 1):
    parser.clear_cache()
    await parser.prefix(ctx(prog[:k]))
cold = time.perf_counter() - t0

print(f'memoized: {memo*1e3:6.0f} ms  ({memo/len(prog)*1e3:.2f} ms/step)')
print(f'cold    : {cold*1e3:6.0f} ms  ({cold/len(prog)*1e3:.2f} ms/step)')
print(f'speedup : {cold/memo:.1f}x')